In [0]:
# CELL 1 — IMPORTS

from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql.window import Window

In [0]:
# CELL 2 — SUPPORTED DQ CONFIGURATION

SUPPORTED_DQ_RULES = [
    "NOT_NULL",
    "UNIQUE",
    "ACCEPTED_VALUES",
    "REGEX",
    "MIN_VALUE",
    "MAX_VALUE",
    "RANGE"
]

SUPPORTED_SEVERITIES = [
    "CRITICAL",
    "HIGH",
    "MEDIUM",
    "LOW"
]

SUPPORTED_FAILURE_ACTIONS = [
    "QUARANTINE",
    "FAIL_DATASET",
    "WARN"
]

APPROVED_RULE_STATUS = "APPROVED"

In [0]:
# CELL 3 — FIND AND SAFELY REFERENCE DATAFRAME COLUMNS

def find_dq_column(df: DataFrame, column_name: str) -> str:
    """Return the actual DataFrame column name using case-insensitive matching."""

    if not isinstance(column_name, str) or not column_name.strip():
        raise ValueError("Column name must be a non-empty string.")

    matches = [
        column
        for column in df.columns
        if column.lower() == column_name.strip().lower()
    ]

    if len(matches) == 0:
        raise ValueError(
            f"Column '{column_name}' does not exist in DataFrame."
        )

    if len(matches) > 1:
        raise ValueError(
            f"Multiple columns match '{column_name}': {matches}"
        )

    return matches[0]


def get_dq_column(df: DataFrame, column_name: str):
    """Return a safely quoted Spark Column reference."""
    actual_column = find_dq_column(df, column_name)
    escaped_name = actual_column.replace("`", "``")
    return F.col(f"`{escaped_name}`")

In [0]:
# CELL 4 — LOAD ACTIVE AND APPROVED DQ RULES

def load_active_dq_rules(
    dq_config_df: DataFrame,
    dataset_name: str
) -> list:
    """Load only rules that are both active and explicitly approved."""

    required_columns = [
        "dataset_name",
        "rule_id",
        "column_name",
        "rule_type",
        "rule_parameters",
        "severity",
        "failure_action",
        "execution_order",
        "is_active",
        "rule_status",
        "description"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in dq_config_df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns in DQ config table: {missing_columns}"
        )

    rules_df = dq_config_df.filter(
        (F.col("dataset_name") == dataset_name) &
        (F.col("is_active") == F.lit(True)) &
        (F.upper(F.trim(F.col("rule_status"))) == APPROVED_RULE_STATUS)
    )

    rules_df = rules_df.orderBy(
        F.col("execution_order").asc(),
        F.col("rule_id").asc()
    )

    rules = [
        row.asDict(recursive=True)
        for row in rules_df.collect()
    ]

    print(f"Dataset: {dataset_name}")
    print(f"Active AND approved DQ rules found: {len(rules)}")

    return rules

In [0]:
# CELL 5 — VALIDATE DQ RULE METADATA

def validate_dq_rules(
    df: DataFrame,
    dq_rules: list
) -> list:
    """Validate rule definitions and referenced input columns before execution."""

    if not isinstance(dq_rules, list):
        raise TypeError("DQ rules must be provided as a list.")

    validated_rules = []
    seen_rule_ids = set()

    for rule in dq_rules:
        if not isinstance(rule, dict):
            raise TypeError("Each DQ rule must be a dictionary.")

        rule_id = rule.get("rule_id")
        column_name = rule.get("column_name")
        rule_type = rule.get("rule_type")
        severity = rule.get("severity")
        failure_action = rule.get("failure_action")
        execution_order = rule.get("execution_order")
        parameters = rule.get("rule_parameters") or {}

        if not isinstance(rule_id, str) or not rule_id.strip():
            raise ValueError("Every DQ rule must have a non-empty rule_id.")
        rule_id = rule_id.strip()

        if rule_id in seen_rule_ids:
            raise ValueError(f"Duplicate DQ rule ID: {rule_id}")
        seen_rule_ids.add(rule_id)

        if not isinstance(rule_type, str) or not rule_type.strip():
            raise ValueError(f"Missing rule_type in {rule_id}.")
        rule_type = rule_type.strip().upper()

        if rule_type not in SUPPORTED_DQ_RULES:
            raise ValueError(
                f"Unsupported DQ rule type '{rule_type}' in {rule_id}."
            )

        if not isinstance(column_name, str) or not column_name.strip():
            raise ValueError(f"Missing column_name in {rule_id}.")
        actual_column = find_dq_column(df, column_name)

        if not isinstance(severity, str):
            raise ValueError(f"Invalid severity '{severity}' in {rule_id}.")
        severity = severity.strip().upper()
        if severity not in SUPPORTED_SEVERITIES:
            raise ValueError(f"Invalid severity '{severity}' in {rule_id}.")

        if not isinstance(failure_action, str):
            raise ValueError(
                f"Invalid failure_action '{failure_action}' in {rule_id}."
            )
        failure_action = failure_action.strip().upper()
        if failure_action not in SUPPORTED_FAILURE_ACTIONS:
            raise ValueError(
                f"Invalid failure_action '{failure_action}' in {rule_id}."
            )

        if (
            not isinstance(execution_order, int)
            or isinstance(execution_order, bool)
            or execution_order < 1
        ):
            raise ValueError(f"Invalid execution_order in {rule_id}.")

        if not isinstance(parameters, dict):
            raise TypeError(
                f"rule_parameters must be a dictionary in {rule_id}."
            )

        # Validate rule-specific parameters before building Spark expressions.
        if rule_type == "ACCEPTED_VALUES":
            accepted_values = parameters.get("accepted_values")
            if isinstance(accepted_values, str):
                accepted_values = [
                    value.strip()
                    for value in accepted_values.split(",")
                    if value.strip()
                ]
            if not isinstance(accepted_values, (list, tuple)) or not accepted_values:
                raise ValueError(
                    f"accepted_values must be a non-empty string or list in {rule_id}."
                )
            parameters = {**parameters, "accepted_values": list(accepted_values)}

        elif rule_type == "REGEX":
            pattern = parameters.get("pattern")
            if not isinstance(pattern, str) or not pattern:
                raise ValueError(f"pattern missing or invalid in {rule_id}.")

        elif rule_type == "MIN_VALUE":
            try:
                parameters = {**parameters, "min_value": float(parameters["min_value"])}
            except (KeyError, TypeError, ValueError):
                raise ValueError(f"A numeric min_value is required in {rule_id}.")

        elif rule_type == "MAX_VALUE":
            try:
                parameters = {**parameters, "max_value": float(parameters["max_value"])}
            except (KeyError, TypeError, ValueError):
                raise ValueError(f"A numeric max_value is required in {rule_id}.")

        elif rule_type == "RANGE":
            try:
                minimum = float(parameters["min_value"])
                maximum = float(parameters["max_value"])
            except (KeyError, TypeError, ValueError):
                raise ValueError(
                    f"Numeric min_value and max_value are required in {rule_id}."
                )
            if minimum > maximum:
                raise ValueError(
                    f"Minimum exceeds maximum in {rule_id}."
                )
            parameters = {
                **parameters,
                "min_value": minimum,
                "max_value": maximum
            }

        validated_rules.append({
            "rule_id": rule_id,
            "column_name": actual_column,
            "rule_type": rule_type,
            "rule_parameters": parameters,
            "severity": severity,
            "failure_action": failure_action,
            "execution_order": execution_order,
            "description": rule.get("description")
        })

    validated_rules.sort(
        key=lambda rule: (rule["execution_order"], rule["rule_id"])
    )

    print(f"Successfully validated {len(validated_rules)} DQ rules.")
    return validated_rules

In [0]:
# CELL 6 — BUILD DQ FAILURE CONDITIONS

def build_dq_condition(
    df: DataFrame,
    rule: dict
):
    """Return a Boolean Spark expression: True means the row failed the rule."""

    column = get_dq_column(df, rule["column_name"])
    rule_type = rule["rule_type"]
    parameters = rule["rule_parameters"]

    if rule_type == "NOT_NULL":
        return column.isNull()

    elif rule_type == "UNIQUE":
        duplicate_count = F.count(F.lit(1)).over(
            Window.partitionBy(column)
        )
        # Nulls are intentionally handled by a separate NOT_NULL rule.
        return column.isNotNull() & (duplicate_count > 1)

    elif rule_type == "ACCEPTED_VALUES":
        accepted_values = parameters["accepted_values"]
        # Nulls are intentionally handled by a separate NOT_NULL rule.
        return F.coalesce(~column.isin(accepted_values), F.lit(False))

    elif rule_type == "REGEX":
        pattern = parameters["pattern"]
        # Nulls are intentionally handled by a separate NOT_NULL rule.
        return F.coalesce(~column.rlike(pattern), F.lit(False))

    elif rule_type in ["MIN_VALUE", "MAX_VALUE", "RANGE"]:
        numeric_column = column.cast("double")
        invalid_cast = column.isNotNull() & numeric_column.isNull()

        if rule_type == "MIN_VALUE":
            minimum = parameters["min_value"]
            return invalid_cast | (
                column.isNotNull() & (numeric_column < minimum)
            )

        elif rule_type == "MAX_VALUE":
            maximum = parameters["max_value"]
            return invalid_cast | (
                column.isNotNull() & (numeric_column > maximum)
            )

        else:  # RANGE
            minimum = parameters["min_value"]
            maximum = parameters["max_value"]
            return invalid_cast | (
                column.isNotNull() & (
                    (numeric_column < minimum) |
                    (numeric_column > maximum)
                )
            )

    raise ValueError(f"Cannot execute rule type: {rule_type}")

In [0]:

# CELL 7 — EXECUTE DQ RULES

def execute_dq_rules(
    df: DataFrame,
    dq_rules: list
) -> dict:
    """Execute DQ rules and split records according to QUARANTINE actions.

    UNIQUE is evaluated within the DataFrame passed to this function (normally
    the current incoming batch). It does not check uniqueness against the full
    existing Silver target table.
    """

    # Validate Spark DataFrame structure instead of relying on isinstance().
    is_spark_dataframe = (
        df is not None
        and hasattr(df, "columns")
        and hasattr(df, "schema")
        and callable(getattr(df, "select", None))
        and callable(getattr(df, "filter", None))
        and callable(getattr(df, "withColumn", None))
    )

    if not is_spark_dataframe:
        raise TypeError(
            "Input must be a Spark DataFrame. "
            f"Received type: {type(df)}"
        )

    validated_rules = validate_dq_rules(df, dq_rules)

    if not validated_rules:
        total_records = df.count()
        print("No active and approved DQ rules found.")
        print(f"Total records passed through: {total_records}")
        return {
            "passed_df": df,
            "quarantined_df": df.limit(0),
            "rule_results": [],
            "total_records": total_records
        }

    working_df = df
    check_columns = []

    # Build one Boolean failure column per rule.
    for index, rule in enumerate(validated_rules):
        condition = build_dq_condition(working_df, rule)
        check_column = f"_silver_dq_check_{index}"

        while check_column in working_df.columns:
            check_column = "_" + check_column

        working_df = working_df.withColumn(check_column, condition)
        check_columns.append(check_column)

    # Calculate the input count and each rule's failure count in one aggregation.
    aggregation_expressions = [
        F.count(F.lit(1)).alias("_total_records")
    ]

    for index, check_column in enumerate(check_columns):
        aggregation_expressions.append(
            F.sum(
                F.when(F.col(check_column), 1).otherwise(0)
            ).alias(f"_failed_{index}")           
        )

    summary = working_df.agg(*aggregation_expressions).first()
    total_records = summary["_total_records"] or 0

    rule_results = []
    quarantine_columns = []
    dataset_failures = []

    # Build complete rule-level results before deciding whether to fail the dataset.
    for index, rule in enumerate(validated_rules):
        failed_records = summary[f"_failed_{index}"] or 0
        passed_records = total_records - failed_records
        failure_percentage = (
            (failed_records / total_records) * 100
            if total_records > 0
            else 0.0
        )

        rule_result = {
            "rule_id": rule["rule_id"],
            "column_name": rule["column_name"],
            "rule_type": rule["rule_type"],
            "severity": rule["severity"],
            "failure_action": rule["failure_action"],
            "total_records": total_records,
            "passed_records": passed_records,
            "failed_records": failed_records,
            "failure_percentage": round(failure_percentage, 2),
            "status": "PASS" if failed_records == 0 else "FAIL"
        }
        rule_results.append(rule_result)

        if rule["failure_action"] == "QUARANTINE":
            quarantine_columns.append(check_columns[index])

        elif (
            rule["failure_action"] == "FAIL_DATASET"
            and failed_records > 0
        ):
            dataset_failures.append(rule_result)

    # FAIL_DATASET takes precedence after all rules have been measured.
    if dataset_failures:
        failure_summary = "; ".join(
            f"{item['rule_id']}: {item['failed_records']} failed"
            for item in dataset_failures
        )
        raise RuntimeError(
            "Dataset validation failed after evaluating all configured rules. "
            f"Failures: {failure_summary}"
        )

    # A row is quarantined once if it fails one or more QUARANTINE rules.
    if quarantine_columns:
        quarantine_condition = F.col(quarantine_columns[0])

        for check_column in quarantine_columns[1:]:
            quarantine_condition = (
                quarantine_condition | F.col(check_column)
            )

        quarantined_df = (
            working_df
            .filter(quarantine_condition)
            .drop(*check_columns)
        )

        passed_df = (
            working_df
            .filter(~quarantine_condition)
            .drop(*check_columns)
        )

    else:
        passed_df = working_df.drop(*check_columns)
        quarantined_df = df.limit(0)

    print(f"Total records: {total_records}")
    print(f"Rules executed: {len(validated_rules)}")
    print(f"Rules with QUARANTINE action: {len(quarantine_columns)}")
    print("DQ execution completed.")

    return {
        "passed_df": passed_df,
        "quarantined_df": quarantined_df,
        "rule_results": rule_results,
        "total_records": total_records
    }


In [0]:
# CELL 8 — DISPLAY DQ RESULTS

def show_dq_results(dq_result: dict):
    """Display rule-level DQ results returned by execute_dq_rules."""

    if not isinstance(dq_result, dict):
        raise TypeError("dq_result must be a dictionary.")

    results = dq_result.get("rule_results", [])

    if not results:
        print("No DQ results available (no active and approved rules).")
        return

    results_df = spark.createDataFrame(results)
    display(results_df.orderBy("rule_id"))

In [0]:
# CELL 9 — CONFIRM EXECUTOR LOAD

print("Silver DQ Executor functions loaded successfully.")
print(f"Supported DQ rules: {SUPPORTED_DQ_RULES}")
print(f"Supported severities: {SUPPORTED_SEVERITIES}")
print(f"Supported failure actions: {SUPPORTED_FAILURE_ACTIONS}")

Silver DQ Executor functions loaded successfully.
Supported DQ rules: ['NOT_NULL', 'UNIQUE', 'ACCEPTED_VALUES', 'REGEX', 'MIN_VALUE', 'MAX_VALUE', 'RANGE']
Supported severities: ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']
Supported failure actions: ['QUARANTINE', 'FAIL_DATASET', 'WARN']
